# Notebook 01 — Ingestão, tratamento e catalogação

**Ambiente:** AWS Glue Studio Notebook (PySpark).  
**Entrada:** CSVs da camada Bronze no S3.  
**Saída:** Parquet na Silver e tabelas no Glue Data Catalog.

Este notebook é interativo e não depende de argumentos de Glue Job. Altere somente a configuração abaixo e execute as células em ordem.

## 1. Configuração do ambiente

In [ ]:
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from pyspark.context import SparkContext
from pyspark.sql import functions as F

BUCKET = 'tech-challenge-018298043465'
DATABASE = 'tech_challenge_db'
YEARS = [2023, 2024, 2025]

sc = SparkContext.getOrCreate()
glue_context = GlueContext(sc)
spark = glue_context.spark_session
print({'bucket': BUCKET, 'database': DATABASE, 'anos': YEARS})

## 2. Funções de ingestão e tratamento

Os nomes originais são preservados porque os SQLs SOT dependem deles. O tratamento remove espaços, converte texto vazio em nulo, identifica colunas totalmente vazias e elimina duplicidades.

In [ ]:
def safe_col(name):
    return F.col(f"`{name.replace('`', '``')}`")

def read_bronze(year):
    path = f's3://{BUCKET}/bases_origem_pesquisas/{year}/'
    return (spark.read.option('header', True).option('quote', '"')
            .option('escape', '"').option('multiLine', True)
            .option('mode', 'PERMISSIVE').csv(path)
            .withColumn('_arquivo_origem', F.input_file_name()))

def clean_dataframe(raw, year):
    if not raw.columns:
        raise ValueError(f'Base {year} sem colunas')
    types = dict(raw.dtypes)
    clean = raw.select(*[(F.when(F.trim(safe_col(c)) == '', None)
        .otherwise(F.trim(safe_col(c))).alias(c)
        if types[c] == 'string' and c != '_arquivo_origem' else safe_col(c).alias(c))
        for c in raw.columns])
    business = [c for c in clean.columns if not c.startswith('_')]
    counts = clean.agg(*[F.count(safe_col(c)).alias(f'nn_{i}')
                         for i, c in enumerate(business)]).first().asDict()
    all_null = [c for i, c in enumerate(business) if counts[f'nn_{i}'] == 0]
    if len(all_null) == len(business):
        raise ValueError(f'Todas as colunas da base {year} estão vazias')
    key = business[0]
    stats = clean.agg(F.count('*').alias('rows'), F.count(safe_col(key)).alias('keys'),
                      F.countDistinct(safe_col(key)).alias('distinct_keys')).first()
    dedup = clean.dropDuplicates([key]) if stats['rows'] == stats['keys'] else clean.dropDuplicates()
    result = (dedup.withColumn('ano_pesquisa', F.lit(int(year)))
              .withColumn('data_processamento', F.current_timestamp()))
    result_count = result.count()
    metrics = {'ano': year, 'linhas_lidas': stats['rows'], 'linhas_gravadas': result_count,
               'duplicidades_removidas': stats['rows'] - result_count, 'chave': key,
               'chaves_distintas': stats['distinct_keys'], 'colunas_totalmente_nulas': all_null}
    return result, metrics

## 3. Processamento dos três anos e controle de qualidade

In [ ]:
processed, quality_log = {}, []
for year in YEARS:
    raw = read_bronze(year)
    if raw.limit(1).count() == 0:
        raise ValueError(f'Base {year} vazia no S3')
    result, metrics = clean_dataframe(raw, year)
    processed[year] = result
    quality_log.append(metrics)
    print(metrics)
    result.select('ano_pesquisa', '_arquivo_origem').show(3, truncate=False)

## 4. Escrita em Parquet e catalogação

O sink do Glue atualiza o Data Catalog e cria `pesquisas_2023`, `pesquisas_2024` e `pesquisas_2025`.

In [ ]:
for year, dataframe in processed.items():
    target = f's3://{BUCKET}/bases_finais_pesquisa/{year}/'
    table = f'pesquisas_{year}'
    dyf = DynamicFrame.fromDF(dataframe, glue_context, f'dyf_{year}')
    sink = glue_context.getSink(path=target, connection_type='s3',
        updateBehavior='UPDATE_IN_DATABASE', partitionKeys=[],
        enableUpdateCatalog=True, transformation_ctx=f'sink_{year}')
    sink.setCatalogInfo(catalogDatabase=DATABASE, catalogTableName=table)
    sink.setFormat('glueparquet')
    sink.writeFrame(dyf)
    print(f'Catalogado: {DATABASE}.{table} -> {target}')

## 5. Verificação final e evidência

In [ ]:
for year in YEARS:
    table = f'{DATABASE}.pesquisas_{year}'
    check = spark.table(table)
    print(table, check.count(), 'linhas', len(check.columns), 'colunas')
spark.createDataFrame(quality_log).orderBy('ano').show(truncate=False)

## Resultado esperado

Os CSVs foram lidos da Bronze, tratados, convertidos para Parquet e catalogados. Guarde prints desta célula e do histórico de execução.